# Gerador de música com gramática generativa

Pipeline: **gramática harmônica** → progressão de acordes → **gramática melódica** por acorde → partes com instrumentos (melodia, harmonia, baixo, bateria) → arquivo **MIDI**.

Estilos disponíveis: `'metroidvania'` (8 bits, tom menor, arpejos rápidos), `'rock'` e `'classica'`. Para trocar, altere `ESTILO` na seção de configuração e use **Kernel → Restart & Run All**.

Além do MIDI, o bloco 12 renderiza um **WAV com timbres 8-bit reais** (ondas de pulso, triângulo e ruído, como o chip de som do NES), que soa igual em qualquer player.

## 1. Importações

In [ ]:
import random
import mido
from music21 import stream, meter, tempo, note, chord, instrument, converter

## 2. Configuração
- `ESTILO`: `'metroidvania'`, `'rock'` ou `'classica'`
- `TOM`: tônica da música (uma das notas de `NOTAS`, ex.: `'C'`, `'G'`, `'Eb'`). No metroidvania a música fica no tom **menor** dessa nota (`'A'` → Lá menor)
- `SEMENTE`: fixe um número para repetir o mesmo resultado; use `None` para variar a cada execução

In [ ]:
ESTILO = 'metroidvania'
TOM = 'A'
SEMENTE = None           # ex.: 42 para resultados reproduzíveis
TEMPOS_POR_ACORDE = 4.0  # um compasso 4/4 por acorde
ARQUIVO_SAIDA = 'music_generated.mid'
ARQUIVO_WAV = 'music_generated_8bit.wav'
BPM = {'metroidvania': 150, 'rock': 120, 'classica': 90}[ESTILO]

random.seed(SEMENTE)

## 3. Teoria básica: notas, intervalos e graus dos acordes
Cada grau (`'1'`, `'3'`, `'5'`…) é contado a partir da fundamental do acorde. Nos acordes maiores, `'b3'` é a *blue note* (terça menor), usada como cor na melodia.

In [ ]:
NOTAS = ['C','Db','D','Eb','E','F','Gb','G','Ab','A','Bb','B']

INTERVALOS = {
    'maior': {'1':0, '2':2, 'b3':3, '3':4, '5':7, '6':9},
    'menor': {'1':0, '2':2, '3':3, '5':7, '6':8},
}

def parse_acorde(nome):
    """'Am' -> ('A', 'menor'); 'C' -> ('C', 'maior')."""
    if nome.endswith('m'):
        return nome[:-1], 'menor'
    return nome, 'maior'

def graus_do_acorde(nome):
    """Mapa grau -> nome da nota para um acorde."""
    fund, qualidade = parse_acorde(nome)
    base = NOTAS.index(fund)
    return {g: NOTAS[(base + st) % 12] for g, st in INTERVALOS[qualidade].items()}

def grau_para_nota(grau, acorde):
    """Consulta o mapa (criado na seção 6). Levanta KeyError se o grau não existe no acorde."""
    return mapa_graus[acorde][grau]

def m21(nome, oitava):
    """Converte para a notação do music21: 'Eb' -> 'E-4'."""
    return nome[0] + nome[1:].replace('b', '-') + str(oitava)

## 4. Gramáticas melódicas
Cada compasso tem duas metades de 2 tempos. Tempos **fortes** usam notas do acorde (1, 3, 5); tempos **fracos** usam notas de passagem (2, 6, b3).

Um apóstrofo depois do grau (`1'`) significa **uma oitava acima**. A gramática de 8 bits usa isso nos arpejos rápidos em semicolcheias, característicos das trilhas de NES.

In [ ]:
gramatica_8bit = {
    'MELODIA': [('COMPASSO', 1.0)],
    'COMPASSO': [('METADE METADE', 1.0)],
    'METADE': [
        ('ARPEJO', 0.25),                                   # 8 semicolcheias
        ('FORTE 0.5 FRACA 0.25 FRACA 0.25 FORTE 1.0', 0.25),
        ('FORTE 1.5 FRACA 0.5', 0.25),                      # nota longa + antecipação
        ('FORTE 0.75 FORTE 0.75 FRACA 0.5', 0.25),          # síncope (3+3+2)
    ],
    'ARPEJO': [
        ("1 0.25 3 0.25 5 0.25 1' 0.25 5 0.25 3 0.25 1 0.25 3 0.25", 0.5),
        ("1 0.25 5 0.25 1' 0.25 5 0.25 3 0.25 5 0.25 1' 0.25 5 0.25", 0.5),
    ],
    'FORTE': [('1', 0.35), ('3', 0.30), ('5', 0.25), ("1'", 0.10)],
    'FRACA': [('2', 0.35), ('5', 0.25), ('3', 0.20), ('6', 0.20)],
}

def gerar(simbolo, regras):
    """Expande um símbolo de uma gramática probabilística até chegar nos terminais."""
    if simbolo not in regras:
        return [simbolo]
    opcoes, pesos = zip(*regras[simbolo])
    escolha = random.choices(opcoes, weights=pesos)[0]
    return [t for s in escolha.split() for t in gerar(s, regras)]

# Teste rápido: sequência de (grau, duração)
gerar('MELODIA', gramatica_8bit)

## 5. Gramáticas harmônicas
- **Clássica**: frase de pergunta (termina em V) e resposta (termina em I), seguindo a lógica Tônica → Subdominante → Dominante.
- **Rock**: blocos de verso/refrão com empréstimos modais (`bVII`, `bVI`).
- **Metroidvania**: tom menor, com as cadências típicas de trilhas de jogos (`i bVI bVII i`), dominante maior da menor harmônica (`V`) e o `bII` frígio para o clima sombrio. Toda frase final termina em `i`, então a música faz *loop* sem emenda.

Para criar regras novas, acrescente tuplas `(expansão, peso)` ou novos símbolos; os pesos não precisam somar 1.

In [ ]:
metroidvania = {
    'PECA':        [('FRASE FRASE_FINAL', 1.0)],
    'FRASE':       [('i bVI bVII i', 0.35), ('i iv bVII bIII', 0.25),
                    ('i bVII bVI V', 0.25), ('i bII i bVII', 0.15)],
    'FRASE_FINAL': [('bVI bVII i i', 0.4), ('iv V i i', 0.4), ('bVI V i i', 0.2)],
}

# Grau romano -> (semitons a partir do tom, sufixo do acorde)
GRAUS_ACORDE = {
    'I': (0, ''), 'ii': (2, 'm'), 'iii': (4, 'm'), 'IV': (5, ''),
    'V': (7, ''), 'vi': (9, 'm'),
    'bIII': (3, ''), 'bVI': (8, ''), 'bVII': (10, ''),
    # graus do tom menor
    'i': (0, 'm'), 'iv': (5, 'm'), 'v': (7, 'm'), 'bII': (1, ''),
}

def grau_para_acorde(grau, tom='C'):
    st, sufixo = GRAUS_ACORDE[grau]
    return NOTAS[(NOTAS.index(tom) + st) % 12] + sufixo

## 6. Gerar a progressão de acordes

In [ ]:
graus = gerar('PECA', 'metroidvania')
progressao = [grau_para_acorde(g, tom=TOM) for g in graus]
mapa_graus = {ac: graus_do_acorde(ac) for ac in progressao}

print('Graus:     ', graus)
print('Progressão:', progressao)

## 7. Instrumentos (General MIDI)
Os números são programas do General MIDI começando em 0. A bateria usa o canal 10 do MIDI (índice 9).

O General MIDI não tem um timbre de NES de verdade: a onda quadrada (programa 80) é a aproximação mais próxima no MIDI. O som 8-bit fiel vem do WAV do bloco 12.

In [ ]:
PROGRAMAS = {
    'metroidvania': {'melodia': 80, 'harmonia': 80, 'baixo': 38}, # Lead 1 (square), square, Synth Bass 1
}
CANAL_BATERIA = 9

def nova_parte(programa, nome, canal=None):
    p = stream.Part()
    p.partName = nome
    inst = instrument.Instrument()
    inst.midiProgram = programa
    inst.partName = nome
    if canal is not None:
        inst.midiChannel = canal
    p.insert(0, inst)
    return p

## 8. Montar as partes: melodia, harmonia, baixo e bateria

In [ ]:
TEM_BATERIA = ESTILO in ('metroidvania')
OITAVA_MELODIA = 5 if ESTILO == 'metroidvania' else 4

melodia  = nova_parte(PROGRAMAS[ESTILO]['melodia'], 'Melodia')
harmonia = nova_parte(PROGRAMAS[ESTILO]['harmonia'], 'Harmonia')
baixo    = nova_parte(PROGRAMAS[ESTILO]['baixo'], 'Baixo')
melodia.insert(0, meter.TimeSignature('4/4'))
melodia.insert(0, tempo.MetronomeMark(number=BPM))

for acorde in progressao:
    # --- Melodia ---
    terminais = gerar('MELODIA', GRAMATICAS_MELODIA[ESTILO])
    usado = 0.0
    for grau, dur in zip(terminais[0::2], terminais[1::2]):
        dur = min(float(dur), TEMPOS_POR_ACORDE - usado)   # não passa do compasso
        if dur <= 0:
            break
        oitava = OITAVA_MELODIA + (1 if grau.endswith("'") else 0)   # 1' = oitava acima
        grau = grau.rstrip("'")
        try:
            nome = grau_para_nota(grau, acorde)
        except KeyError:                                     # ex.: 'b3' em acorde menor
            nome = grau_para_nota('1', acorde)
        melodia.append(note.Note(m21(nome, oitava), quarterLength=dur))
        usado += dur
    if usado < TEMPOS_POR_ACORDE:
        melodia.append(note.Rest(quarterLength=TEMPOS_POR_ACORDE - usado))

    # --- Harmonia ---
        # O chip do NES não toca acordes: a harmonia vira arpejo em semicolcheias
    ciclo = ['1', '3', '5', '3']
    for i in range(16):
        harmonia.append(note.Note(m21(grau_para_nota(ciclo[i % 4], acorde), 4), quarterLength=0.25))

    # --- Baixo ---
    raiz = grau_para_nota('1', acorde)
    for i in range(8):                       # oitavas alternadas, típico do canal triângulo
        baixo.append(note.Note(m21(raiz, 2 if i % 2 == 0 else 3), quarterLength=0.5))

partes = [melodia, harmonia, baixo]

bateria = nova_parte(0, 'Bateria', canal=CANAL_BATERIA)
BUMBO, CAIXA, CHIMBAL = 36, 38, 42
for _ in progressao:
    for i in range(8):                      # 8 colcheias por compasso
        golpes = [CHIMBAL]
        if i in (0, 4):
            golpes.append(BUMBO)            # tempos 1 e 3
        if i in (2, 6):
            golpes.append(CAIXA)            # tempos 2 e 4
        bateria.append(chord.Chord([note.Note(midi=g) for g in golpes], quarterLength=0.5))
partes.append(bateria)

partitura = stream.Score(partes)

## 9. Salvar o MIDI
Depois de gravar, o arquivo passa por uma correção com `mido`: garante a troca de instrumento em cada trilha e coloca a bateria no canal 10, independentemente de como a versão do music21 distribuiu os canais.

In [ ]:
partitura.write('midi', fp=ARQUIVO_SAIDA)

def corrigir_canais(caminho, programas, tem_bateria):
    """Força um canal e um program_change por trilha; a última trilha vira bateria (canal 10)."""
    mid = mido.MidiFile(caminho)
    trilhas_notas = [t for t in mid.tracks if any(m.type == 'note_on' for m in t)]
    for idx, trilha in enumerate(trilhas_notas):
        eh_bateria = tem_bateria and idx == len(trilhas_notas) - 1
        canal = CANAL_BATERIA if eh_bateria else idx
        programa = 0 if eh_bateria else programas[idx]
        novas = [m for m in trilha if m.type != 'program_change']
        for m in novas:
            if hasattr(m, 'channel'):
                m.channel = canal
        insere = next((i for i, m in enumerate(novas) if not m.is_meta), len(novas))
        novas.insert(insere, mido.Message('program_change', channel=canal, program=programa, time=0))
        trilha[:] = novas
    mid.save(caminho)

ordem = [PROGRAMAS[ESTILO][k] for k in ('melodia', 'harmonia', 'baixo')]
corrigir_canais(ARQUIVO_SAIDA, ordem, tem_bateria=TEM_BATERIA)
print('MIDI salvo em', ARQUIVO_SAIDA)

## 10. Conferir os instrumentos gravados no arquivo

In [ ]:
for i, trilha in enumerate(mido.MidiFile(ARQUIVO_SAIDA).tracks):
    programas = [(m.channel + 1, m.program) for m in trilha if m.type == 'program_change']
    notas = sum(1 for m in trilha if m.type == 'note_on' and m.velocity > 0)
    print(f'Trilha {i}: (canal, programa) = {programas}, notas = {notas}')

## 11. Ouvir no notebook
O player do notebook roda no navegador e às vezes não carrega todos os timbres. Se soar tudo como piano, abra o arquivo `.mid` no MuseScore, VLC ou numa DAW (Reaper, LMMS, GarageBand).

In [ ]:
converter.parse(ARQUIVO_SAIDA).show('midi')

## 12. Renderizar em 8 bits de verdade (WAV)
Sintetiza a partitura com as mesmas formas de onda do chip de som do NES, sem depender de soundfont:
- **Melodia**: onda de pulso 25% (o som "de NES" clássico)
- **Harmonia**: pulso 12,5% (mais fino, não briga com a melodia)
- **Baixo**: triângulo com 16 níveis de volume, como no hardware original
- **Bateria**: ruído (caixa e chimbal) e triângulo com queda de afinação (bumbo)

Funciona com qualquer `ESTILO`: o rock e a clássica também viram versões 8-bit.

In [ ]:
import numpy as np
import wave
from IPython.display import Audio

SR = 44100
TIMBRES_8BIT = {
    'Melodia':  ('pulso25', 0.30),
    'Harmonia': ('pulso12', 0.15),
    'Baixo':    ('triangulo', 0.40),
}

def onda(tipo, freq, n):
    fase = (np.arange(n) * freq / SR) % 1.0
    if tipo == 'pulso50':
        return np.where(fase < 0.5, 1.0, -1.0)
    if tipo == 'pulso25':
        return np.where(fase < 0.25, 1.0, -1.0)
    if tipo == 'pulso12':
        return np.where(fase < 0.125, 1.0, -1.0)
    if tipo == 'triangulo':
        tri = 4 * np.abs(fase - 0.5) - 1
        return np.round(tri * 7.5) / 7.5          # 16 degraus, como no NES
    raise ValueError(tipo)

def envelope(n, ataque=0.002, soltura=0.01):
    env = np.ones(n)
    a, r = min(n, int(ataque * SR)), min(n, int(soltura * SR))
    if a: env[:a] = np.linspace(0, 1, a)
    if r: env[-r:] *= np.linspace(1, 0, r)
    return env

def tambor(midi_num):
    if midi_num == 36:                                    # bumbo: triângulo caindo de 150 para 50 Hz
        n = int(0.12 * SR)
        freq = np.linspace(150, 50, n)
        fase = np.cumsum(freq / SR) % 1.0
        return 0.6 * (4 * np.abs(fase - 0.5) - 1) * np.linspace(1, 0, n)
    dur, vol = (0.12, 0.35) if midi_num == 38 else (0.03, 0.12)   # caixa / chimbal
    n = int(dur * SR)
    return vol * np.random.uniform(-1, 1, n) * np.exp(-np.linspace(0, 5, n))

def renderizar_8bit(partitura, bpm, caminho, gate=0.9):
    seg_tempo = 60 / bpm
    mix = np.zeros(int(partitura.highestTime * seg_tempo * SR) + SR)
    for parte in partitura.parts:
        for el in parte.flatten().notes:
            ini = int(el.offset * seg_tempo * SR)
            if parte.partName == 'Bateria':
                for p in el.pitches:
                    s = tambor(p.midi)
                    mix[ini:ini + len(s)] += s
                continue
            tipo, vol = TIMBRES_8BIT[parte.partName]
            n = max(1, int(el.quarterLength * seg_tempo * SR * gate))   # gate < 1: notas destacadas
            for p in el.pitches:
                f = 440 * 2 ** ((p.midi - 69) / 12)
                mix[ini:ini + n] += vol * onda(tipo, f, n) * envelope(n)
    mix = 0.9 * mix / max(1e-9, np.abs(mix).max())
    with wave.open(caminho, 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(SR)
        w.writeframes((mix * 32767).astype(np.int16).tobytes())
    return caminho

renderizar_8bit(partitura, BPM, ARQUIVO_WAV)
Audio(ARQUIVO_WAV)

## 13. (Opcional) Ouvir um MIDI de referência

In [ ]:
musica_ref = converter.parse('./samples/Goodbye Yellow Brick Road Elton John.mid')
musica_ref.show('midi')